# 🔗 Notebook 3: End-to-End Pipeline Comparison
Combine Segmentation + Classification models into 3 full pipelines and compare results.
> **Run Notebooks 1 & 2 first** to generate the saved model weights.

In [ ]:
import os, glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image, ImageDraw
from pathlib import Path
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision import transforms

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATASET_BASE = r"c:\Users\DELL\Desktop\Vinh Hoang\Master Program\Học sâu\Project\Dataset"
# DATASET_BASE = "/content/drive/MyDrive/Project/Dataset"

SEG_RESULTS = os.path.join(os.path.dirname(DATASET_BASE), "results", "segmentation")
CLS_RESULTS = os.path.join(os.path.dirname(DATASET_BASE), "results", "classification")
cls_dir     = os.path.join(DATASET_BASE, "Elsafty_RBCs_for_Classification", "Cropped images")
# CHỈ lấy những thư mục thực sự chứa ảnh .png (giống như Notebook 2 đã làm)
CLASSES = sorted([d.name for d in Path(cls_dir).iterdir() if d.is_dir() and any(d.glob("*.png"))])
IMG_SIZE = 80
print(f"Device: {DEVICE}")
print(f"Classes ({len(CLASSES)}): {CLASSES}")


In [ ]:
# Re-define model classes (copy architecture from Notebooks 1 & 2)
# ── Seg models ──
class CBAM(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.ca = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(channels, max(1, channels//reduction), 1, bias=False), nn.ReLU(inplace=True), nn.Conv2d(max(1, channels//reduction), channels, 1, bias=False), nn.Sigmoid())
        self.sa = nn.Sequential(nn.Conv2d(2, 1, 7, padding=3, bias=False), nn.Sigmoid())
    def forward(self, x):
        x = x * self.ca(x); max_o, _ = torch.max(x, dim=1, keepdim=True); avg_o = torch.mean(x, dim=1, keepdim=True)
        return x * self.sa(torch.cat([max_o, avg_o], dim=1))

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch,out_ch,3,padding=1,bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch,out_ch,3,padding=1,bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
        self.cbam = CBAM(out_ch)
    def forward(self,x): return self.cbam(self.net(x))

class MiniUNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, features=[16,32,64]):
        super().__init__()
        self.encoders=nn.ModuleList(); self.pools=nn.ModuleList()
        self.decoders=nn.ModuleList(); self.upconvs=nn.ModuleList()
        prev=in_ch
        for f in features:
            self.encoders.append(DoubleConv(prev,f)); self.pools.append(nn.MaxPool2d(2)); prev=f
        self.bottleneck=DoubleConv(prev,prev*2); prev=prev*2
        for f in reversed(features):
            self.upconvs.append(nn.ConvTranspose2d(prev,f,2,2))
            self.decoders.append(DoubleConv(f*2,f)); prev=f
        self.head=nn.Conv2d(prev,out_ch,1)
    def forward(self,x):
        skips=[]
        for enc,pool in zip(self.encoders,self.pools):
            x=enc(x); skips.append(x); x=pool(x)
        x=self.bottleneck(x); skips=skips[::-1]
        for up,dec,skip in zip(self.upconvs,self.decoders,skips):
            x=up(x)
            if x.shape!=skip.shape: x=F.interpolate(x,size=skip.shape[2:])
            x=dec(torch.cat([skip,x],dim=1))
        return self.head(x)

# ── Cls models ──
class CNNClassifier(nn.Module):
    def __init__(self,n=9):
        super().__init__()
        self.layer1=nn.Sequential(nn.Conv2d(3,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(),nn.MaxPool2d(2))
        self.cbam1=CBAM(32)
        self.layer2=nn.Sequential(nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(),nn.MaxPool2d(2))
        self.cbam2=CBAM(64)
        self.layer3=nn.Sequential(nn.Conv2d(64,128,3,padding=1),nn.BatchNorm2d(128),nn.ReLU(),nn.MaxPool2d(2))
        self.cbam3=CBAM(128)
        self.layer4=nn.Sequential(nn.Conv2d(128,256,3,padding=1),nn.BatchNorm2d(256),nn.ReLU(),nn.AdaptiveAvgPool2d(1))
        self.head=nn.Sequential(nn.Flatten(),nn.Dropout(0.4),nn.Linear(256,n))
    def forward(self,x): return self.head(self.layer4(self.cbam3(self.layer3(self.cbam2(self.layer2(self.cbam1(self.layer1(x))))))))

print("Model classes defined ✅")


In [ ]:
# ── Load saved weights ────────────────────────────────────────────────────────
def load_model(cls, path, device=DEVICE):
    m = cls().to(device)
    m.load_state_dict(torch.load(path, map_location=device))
    m.eval()
    return m

seg_cnn = load_model(MiniUNet, f"{SEG_RESULTS}/CNN_UNet_best.pt")
cls_cnn = load_model(CNNClassifier, f"{CLS_RESULTS}/CNN_Cls_best.pt")
print("Loaded CNN pipeline ✅")
# Add RNN & ViT models here similarly after pasting architectures from NB1 & NB2


In [ ]:
# ── Pipeline inference function ───────────────────────────────────────────────
img_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

def run_pipeline(seg_model, cls_model, crop_img_pil):
    """
    Given an 80x80 PIL crop:
      1. Segmentation → binary mask
      2. Apply mask → segmented cell
      3. Classification → class probabilities
    """
    tensor = img_tf(crop_img_pil).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        mask_logit = seg_model(tensor)
        mask       = (torch.sigmoid(mask_logit) > 0.5).float()
        logits     = cls_model(tensor)
        probs      = F.softmax(logits, dim=1).squeeze().cpu().numpy()
    pred_cls   = CLASSES[probs.argmax()]
    confidence = probs.max()
    mask_np    = mask.squeeze().cpu().numpy()
    return pred_cls, confidence, mask_np, probs


def compare_pipelines(pipelines, test_img_paths, n=5):
    """Show results of all pipelines on n sample images."""
    samples = test_img_paths[:n]
    n_pipes = len(pipelines)
    fig, axes = plt.subplots(n, 1 + n_pipes, figsize=(4 * (1+n_pipes), 3.5 * n))

    for i, path in enumerate(samples):
        img = Image.open(path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
        axes[i][0].imshow(img); axes[i][0].set_title("Input"); axes[i][0].axis("off")

        for j, (name, seg, cls) in enumerate(pipelines):
            pred_cls, conf, mask, probs = run_pipeline(seg, cls, img)
            ax = axes[i][1+j]
            ax.imshow(mask, cmap="gray")
            ax.set_title(f"{name}\n{pred_cls}\n({conf:.0%})", fontsize=8)
            ax.axis("off")

    plt.suptitle("End-to-End Pipeline Comparison: CNN vs RNN vs Transformer",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(os.path.dirname(DATASET_BASE), "results", "e2e_comparison.png"), dpi=150)
    plt.show()

print("Pipeline functions defined ✅")


In [ ]:
# ── Run comparison on test crops ──────────────────────────────────────────────
test_crops = sorted(glob.glob(
    os.path.join(DATASET_BASE, "Elsafty_RBCs_for_Classification", "Cropped images", "**", "*.png"),
    recursive=True
))[:50]  # use first 50 as demo images

pipelines = [
    ("CNN", seg_cnn, cls_cnn),
    # ("RNN", seg_rnn, cls_rnn),   # uncomment after loading RNN pipeline
    # ("ViT", seg_vit, cls_vit),   # uncomment after loading ViT pipeline
]

compare_pipelines(pipelines, test_crops, n=5)


In [ ]:
# ── Final summary table ───────────────────────────────────────────────────────
import pandas as pd

summary = {
    "Pipeline": ["CNN", "RNN", "Transformer"],
    "Seg Model": ["Mini U-Net", "ConvLSTM U-Net", "ViT-UNet"],
    "Cls Model": ["Custom CNN", "Bi-GRU", "Custom ViT"],
    # Fill in from Notebooks 1 & 2 results
    "Seg Dice":  [None, None, None],
    "Seg IoU":   [None, None, None],
    "Cls Acc":   [None, None, None],
    "Cls F1":    [None, None, None],
}

df = pd.DataFrame(summary)
print(df.to_string(index=False))
# After filling in metrics:
# df.to_csv(os.path.join(os.path.dirname(DATASET_BASE), "results", "final_summary.csv"), index=False)
